<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-07/blob/main/hw_07_rdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 7 — Time Series and Semi-Structured Data for ML

## 1. Setup and NYC Taxi Data Loading

In [12]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

In [15]:
%pip install -q "psycopg[binary]" sqlalchemy pandas pyarrow requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 15.9 MB/s eta 0:00:00


In [16]:
import os
import sys
import pandas as pd
import pgserver

from urllib.request import urlretrieve
from sqlalchemy import create_engine, text

print("Python:", sys.version)

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 1.1 NYC Taxi Data Loading

Dataset: NYC TLC Yellow Taxi Trip Records, March 2024.
Source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [2]:
URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2024-03.parquet"
)

parquet_path = "/content/yellow_tripdata_2024-03.parquet"

urlretrieve(URL, parquet_path)

columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count",
]

df = pd.read_parquet(parquet_path, columns=columns).head(500_000)

print("Loaded rows:", len(df))
print("Shape:", df.shape)

display(df.head())

Loaded rows: 500000
Shape: (500000, 7)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,passenger_count
0,2024-03-01 00:18:51,2024-03-01 00:23:45,142,239,1.30,8.6,0.0
1,2024-03-01 00:26:00,2024-03-01 00:29:06,238,24,1.10,7.2,0.0
2,2024-03-01 00:09:22,2024-03-01 00:15:24,263,75,0.86,7.9,1.0
3,2024-03-01 00:33:45,2024-03-01 00:39:34,164,162,0.82,7.9,1.0
4,2024-03-01 00:05:43,2024-03-01 00:26:22,263,7,4.90,25.4,0.0
